# Tutorial 8: Choosing the Next Word, Greedily or with a Beam

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/tutorials/08-inference-and-beamsearch.ipynb)

**Reading. About 35 minutes.**

A trained model gives a probability for every possible next word; decoding is how a translation gets chosen from them. This notebook writes the two standard ways out in full, greedy search and beam search, and runs them side by side. The choice changes the translations, and so the score: that is why A9 decodes your A8 and A9 models the same way, greedily, before comparing them. It ends with two ways of *not* choosing the most likely translation, which back-translation uses.

By the end you will be able to:

1. explain how greedy search and beam search each choose a translation, having written both out and run them on the same sentences;
2. say what each decoding setting changes, `beam_size`, the length penalty `alpha` and the maximum output length, and why a comparison between two models must hold all three fixed;
3. score translations with the library's evaluation functions, and read the score with its signature;
4. say what sampling and noised beam search produce, and why back-translation might want them when scoring never does.

These serve Lecture 10's outcome *explain greedy against beam search and why a comparison must pin decoding*.

**Running in Google Colab?** Run the next two cells: the first installs TorchLingo, the second checks the setup. There is nothing to uncomment, and no GPU is needed: the notebook trains its own tiny model in a few seconds.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**TorchLingo tutorial** — assigned reading for Lecture 10 (*Overview of MT Quality Estimation*). A head start on **A13**, **A14**.

In [1]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo

In [2]:
# Check the setup: the TorchLingo and PyTorch versions and the device, printed here so that
# anything wrong shows up now rather than ten cells from now.
from torchlingo.colab import setup

env = setup()
device = env.device

import torch
import torch.nn.functional as F

TorchLingo 0.2.8 (editable checkout) | PyTorch 2.13.0 | device cpu | not in Colab


In [3]:
# Setup: run this cell, no need to read it.
# A tiny model trained here so this notebook runs on its own: ten travel phrases, a few
# seconds of training. It memorizes them, which matters later.
import contextlib
import io
import tempfile
from pathlib import Path

import pandas as pd
from torch.utils.data import DataLoader

from torchlingo.config import Config
from torchlingo.data_processing import NMTDataset, collate_fn
from torchlingo.models import SimpleTransformer
from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer

phrases = [
    ("where is the station", "dónde está la estación"), ("the train is late", "el tren llega tarde"),
    ("one ticket please", "un billete por favor"), ("the bus leaves now", "el autobús sale ahora"),
    ("turn left here", "gira a la izquierda aquí"), ("turn right here", "gira a la derecha aquí"),
    ("the hotel is near", "el hotel está cerca"), ("the beach is far", "la playa está lejos"),
    ("we need a taxi", "necesitamos un taxi"), ("my bag is heavy", "mi maleta pesa mucho"),
]
# A temporary folder, so running the notebook leaves no file behind.
train_path = Path(tempfile.mkdtemp()) / "travel_phrases.tsv"
pd.DataFrame(phrases * 20, columns=["src", "tgt"]).to_csv(train_path, sep="\t", index=False)
dataset = NMTDataset(train_path)
src_vocab, tgt_vocab = dataset.src_vocab, dataset.tgt_vocab

torch.manual_seed(0)
config = Config(d_model=64, n_heads=4, num_encoder_layers=2, num_decoder_layers=2,
                d_ff=128, dropout=0.1, batch_size=8, learning_rate=1e-3)
model = SimpleTransformer(src_vocab_size=len(src_vocab), tgt_vocab_size=len(tgt_vocab),
                          config=config).to(device)
loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn,
                    num_workers=0)
with contextlib.redirect_stdout(io.StringIO()):  # the per-epoch log, not needed here
    result = train_model(model, loader, num_epochs=40, device=device, config=config,
                         gradient_clip=1.0, log_every=0,
                         checkpointer=TrainingCheckpointer("phrases", checkpoint_dir=tempfile.mkdtemp(), verbose=False))
model.eval()
print(f"Trained a tiny model on {len(phrases)} phrases: "
      f"final loss {result.train_losses[-1]:.3f}")

Trained a tiny model on 10 phrases: final loss 0.712


## Part 1: Greedy decoding

Greedy decoding picks the single most likely token at each step, and stops at `<eos>` or at `max_len`, the most tokens it may write. That limit is the first of the three decoding settings, and the cell after the function shows it at work.

In [4]:
def greedy_decode(model, src_sentence, src_vocab, tgt_vocab, device, max_len=20):
    """Generate translation using greedy decoding.
    
    At each step, pick the single most likely next token.
    """
    model.eval()
    
    # Encode source
    src_indices = src_vocab.encode(src_sentence, add_special_tokens=True)
    src_tensor = torch.tensor([src_indices]).to(device)
    
    with torch.no_grad():
        memory = model.encode(src_tensor)
    
    # Decode
    output_indices = [tgt_vocab.sos_idx]
    
    for _ in range(max_len):
        tgt_tensor = torch.tensor([output_indices]).to(device)
        
        with torch.no_grad():
            logits = model.decode(tgt_tensor, memory)
        
        # Greedy: pick argmax
        next_token = logits[0, -1, :].argmax().item()
        output_indices.append(next_token)
        
        if next_token == tgt_vocab.eos_idx:
            break
    
    return tgt_vocab.decode(output_indices, skip_special_tokens=True)

In [5]:
# Test greedy decoding
test_sentence = "the train is late"
translation = greedy_decode(model, test_sentence, src_vocab, tgt_vocab, device)
print(f"Greedy:          '{test_sentence}' → '{translation}'")

# The maximum length at work: too small a limit cuts the translation off mid-sentence,
# with nothing to say so. On real sentences, set it from your longest target.
cut = greedy_decode(model, test_sentence, src_vocab, tgt_vocab, device, max_len=1)
print(f"Greedy, max_len=1: '{test_sentence}' → '{cut}'")

Greedy:          'the train is late' → 'el tren llega tarde'
Greedy, max_len=1: 'the train is late' → 'el'


### Greedy Limitations

Greedy decoding can get stuck in suboptimal paths:

```
Step 1: P("el") = 0.4, P("la") = 0.35, P("un") = 0.25
        → Pick "el" (highest)
        
Step 2: P("tren"|"el") = 0.3, P("hotel"|"el") = 0.25, ...
        → But maybe "la estación" would have been better overall!
```

Greedy only considers one path—it can't backtrack.

## Part 2: Beam Search

Beam search keeps track of multiple hypotheses ("beams") and picks the best complete sequence. It is written in two cells: `expand` takes one step, growing every hypothesis by its best next tokens, and `beam_search_decode` repeats that step, prunes back to `beam_size` each time, and makes the final choice with the length penalty.

In [6]:
def expand(beams, memory, model, beam_size, eos_idx, device):
    """One step of beam search: extend each unfinished hypothesis by its best next tokens.

    A hypothesis is (tokens so far, total log-probability). Each unfinished one proposes
    its beam_size most likely continuations; one that already ends in <eos> is set aside.

    Returns:
        The candidates for the next step, and the hypotheses that had already finished.
    """
    candidates, finished = [], []
    for seq, score in beams:
        if seq[-1] == eos_idx:
            finished.append((seq, score))
            continue

        tgt_tensor = torch.tensor([seq]).to(device)
        with torch.no_grad():
            logits = model.decode(tgt_tensor, memory)
        log_probs = F.log_softmax(logits[0, -1, :], dim=-1)

        # Adding log-probabilities is multiplying probabilities, without the underflow.
        topk_log_probs, topk_indices = log_probs.topk(beam_size)
        for log_prob, idx in zip(topk_log_probs, topk_indices):
            candidates.append((seq + [idx.item()], score + log_prob.item()))
    return candidates, finished

In [7]:
def beam_search_decode(
    model, src_sentence, src_vocab, tgt_vocab, device,
    beam_size=3, max_len=20, length_penalty=0.6
):
    """Generate a translation with beam search, keeping beam_size hypotheses per step.

    Args:
        beam_size: How many hypotheses survive each step. 1 is greedy decoding.
        length_penalty: alpha. The final choice divides each total log-probability by
            length ** alpha, so a longer output is not ruled out merely for having more
            (negative) terms in its sum. 0 turns it off.
    """
    model.eval()
    src_indices = src_vocab.encode(src_sentence, add_special_tokens=True)
    src_tensor = torch.tensor([src_indices]).to(device)
    with torch.no_grad():
        memory = model.encode(src_tensor)

    beams = [([tgt_vocab.sos_idx], 0.0)]
    completed = []
    for _ in range(max_len):
        candidates, finished = expand(
            beams, memory, model, beam_size, tgt_vocab.eos_idx, device
        )
        completed.extend(finished)
        # Prune: only the beam_size best by total log-probability go on.
        candidates.sort(key=lambda x: x[1], reverse=True)
        beams = candidates[:beam_size]
        if not beams:
            break
    completed.extend(beams)  # whatever was still running when max_len ran out

    best_seq, _ = max(completed, key=lambda x: x[1] / len(x[0]) ** length_penalty)
    return tgt_vocab.decode(best_seq, skip_special_tokens=True)

In [8]:
# Test beam search
test_sentence = "where is the station"

greedy_result = greedy_decode(model, test_sentence, src_vocab, tgt_vocab, device)
beam_result = beam_search_decode(model, test_sentence, src_vocab, tgt_vocab, device, beam_size=3)

print(f"Input:  '{test_sentence}'")
print(f"Greedy: '{greedy_result}'")
print(f"Beam-3: '{beam_result}'")

Input:  'where is the station'
Greedy: 'dónde está la estación'
Beam-3: 'dónde está la estación'


## Part 3: Comparing Strategies

In [9]:
# Compare on multiple sentences
test_sentences = [
    "where is the station",
    "one ticket please",
    "turn left here",
    "we need a taxi",
    "the beach is far",
]

print(f"{'Input':<22} {'Greedy':<26} {'Beam-3':<26}")
print("-" * 74)

for src in test_sentences:
    greedy = greedy_decode(model, src, src_vocab, tgt_vocab, device)
    beam = beam_search_decode(model, src, src_vocab, tgt_vocab, device)
    print(f"{src:<22} {greedy:<26} {beam:<26}")

Input                  Greedy                     Beam-3                    
--------------------------------------------------------------------------
where is the station   dónde está la estación     dónde está la estación    
one ticket please      un billete por favor       un billete por favor      
turn left here         gira a la izquierda aquí   gira a la izquierda aquí  


we need a taxi         necesitamos un taxi        necesitamos un taxi       


the beach is far       la playa está lejos        la playa está lejos       


In [10]:
# Effect of beam size -- or rather, the absence of one.
test_sentence = "where is the station"

results = {}
for beam_size in [1, 2, 3, 5, 10]:
    results[beam_size] = beam_search_decode(
        model, test_sentence, src_vocab, tgt_vocab, device,
        beam_size=beam_size
    )

print(f"Input: '{test_sentence}'")
print("-" * 40)
for beam_size, result in results.items():
    print(f"Beam-{beam_size:2d}: '{result}'")

# The point of this cell is that every row is the same, so the notebook asserts
# it rather than leaving you to notice. If this ever fires, the model has become
# uncertain enough for beam width to matter and the explanation below it is no
# longer the right one -- which is worth knowing loudly.
distinct = set(results.values())
assert len(distinct) == 1, (
    f"expected one translation across all beam sizes, got {len(distinct)}: "
    f"{distinct}. The model is no longer decisive, so the discussion below "
    "this cell needs rewriting."
)

print()
print(f"{len(results)} beam sizes, {len(distinct)} distinct translation.")
print("Widening the beam from 1 to 10 changed nothing at all.")

Input: 'where is the station'
----------------------------------------
Beam- 1: 'dónde está la estación'
Beam- 2: 'dónde está la estación'
Beam- 3: 'dónde está la estación'
Beam- 5: 'dónde está la estación'
Beam-10: 'dónde está la estación'

5 beam sizes, 1 distinct translation.
Widening the beam from 1 to 10 changed nothing at all.


In [11]:
# The third setting, the length penalty alpha, on the same sentence and a beam of 5.
for alpha in [0.0, 0.6, 1.0]:
    out = beam_search_decode(
        model, test_sentence, src_vocab, tgt_vocab, device,
        beam_size=5, length_penalty=alpha,
    )
    print(f"alpha={alpha}: '{out}'")

alpha=0.0: 'dónde está la estación'
alpha=0.6: 'dónde está la estación'
alpha=1.0: 'dónde está la estación'


### A measurement that tells you nothing

Five identical rows for the beam size, and three for alpha. It is tempting to read that as
"these settings do not matter," and that would be the wrong lesson from a correct
observation.

What it actually means is that **this experiment cannot answer the question.** The model
above trained on ten phrases and we are decoding one of them. It has memorized the
answer, so it is certain, and a certain model gives beam search nothing to search. You
would see the same rows whether beam width mattered enormously or not at all.

This is worth more than the result it failed to produce. A sweep that returns the same
value for every setting is telling you about your *setup*, not about the parameter:

- the model may be too certain, as here
- the parameter may not be reaching the code you think it is
- the effect may be real but smaller than the resolution of your measurement

Distinguishing those is most of experimental debugging. The first question is never
"what does this tell me about beam size" but "could this experiment have detected an
effect if there were one?"

Here it could not. To measure the real thing you need a model that is genuinely unsure —
one trained on real data, translating a sentence it has never seen. That measurement is
in [Decoding](../concepts/decoding.md), run on the pretrained model from
[Tutorial 7](07-real-translations.ipynb) across several held-out samples. In short:
greedy to *any* beam width is the large gain, quality peaks around beam 3 to 5, and
widening further makes it measurably **worse**.

### Watching the search prune

Every table so far shows only the **winner**. Beam search spends its whole budget on
hypotheses that lose, and the case for paying that cost is entirely about paths greedy
would never reach — which the output never shows you.

The library's `beam_search_decode` will record them if you hand it a list:

```python
trace = []
tokens = beam_search_decode(model, src, beam_size=3, trace=trace)
```

Tracing changes nothing about the result. It only writes down what happened.

In [12]:
from torchlingo.inference import beam_search_decode as library_beam_search
from torchlingo.visualization import format_beam_search

sentence = "turn right here"
ids = src_vocab.encode(sentence, add_special_tokens=True)

trace = []
tokens = library_beam_search(
    model, torch.tensor([ids]).to(device), beam_size=3, max_len=8, trace=trace
)

print(f"{sentence!r} -> {tgt_vocab.decode(tokens, skip_special_tokens=True)!r}")
print()
print(format_beam_search(trace, itos=tgt_vocab.idx2token, winner=tokens, top=3, max_steps=4))
print()
print("  >  kept, and on the path that eventually won")
print("  +  kept into the next step")
print("  .  pruned here")

'turn right here' -> 'gira a la derecha aquí'

step 0
  > -0.029  <sos> gira
  + -5.401  <sos> el
  + -5.545  <sos> aquí
step 1
  > -0.065  <sos> gira a
  + -4.694  <sos> gira aquí
  + -4.787  <sos> gira derecha
    ... 6 more considered
step 2
  > -0.087  <sos> gira a la
  + -4.526  <sos> gira aquí <eos>
  + -4.717  <sos> gira derecha aquí
    ... 6 more considered
step 3
  > -0.102  <sos> gira a la derecha
  + -4.297  <sos> gira a la taxi
  + -4.485  <sos> gira a la izquierda
    ... 3 more considered

  >  kept, and on the path that eventually won
  +  kept into the next step
  .  pruned here


**The `>` is at the top of every step, and it is not close** — roughly `-0.03` against
`-5` for the runner-up. This model is certain, so beam search follows exactly the path
greedy would have taken, and every one of those extra hypotheses was wasted work.

That is the honest result on this model, and it is worth sitting with rather than
skipping past: **beam search bought us nothing here.** The model trained on ten
phrases and we are decoding one of them, so it has memorized the answer. A memorized
model has no uncertainty for beam search to exploit.

The interesting case is a `>` sitting *below* a `+`: the eventual winner ranked second or
third at that step and survived only because the beam was wide enough to carry it. Greedy
would have committed to the higher-scoring option and had no way back. That is the
situation beam search exists for, and you need a model that is genuinely unsure — one
trained on real data, translating a sentence it has never seen — to produce it.

So the honest summary of this section is: you now have the instrument, and this model is
the wrong patient for it. Point it at a real model and look for the crossover.

## Part 4: Does this match the library?

The two functions above are written to be *read*. TorchLingo ships its own
versions in `torchlingo.inference`, and you should use those in real work:

```python
from torchlingo.inference import beam_search_decode, greedy_decode
```

Whenever a tutorial reimplements something the library provides, the two can
drift apart without anyone noticing — and then you learn one algorithm while the
library runs another. So let's check, rather than assume.

**Three places the version above is deliberately simpler:**

| | This notebook | `torchlingo.inference` |
| --- | --- | --- |
| **Pruning** | keeps the top `beam_size` by raw cumulative score | ranks by the *length-normalized* score |
| **Length penalty** | `score / length ** α` | `score / ((5 + length) / 6) ** α` (Wu et al., 2016) |
| **Ties** | `log_probs.topk(k)` | `_canonical_topk`, a documented tie-breaking rule |

The third is the one worth understanding. `torch.topk` does **not** promise any
particular order among equally-scored elements, so a beam search built on it can
return different output on different devices — for the same model and the same
input. The library defines an explicit rule instead: *prefer the higher score;
among exactly equal scores prefer the lower token IDs, position by position.*
Ties are common early in training and with padded inputs, so this matters more
than it sounds.

None of those differences changes the answer on this model, which is decisive
enough never to hit a tie. The cell below proves that — and will fail loudly if
it ever stops being true.

In [13]:
from torchlingo.inference import beam_search_decode as library_beam_search
from torchlingo.inference import greedy_decode as library_greedy_decode


def library_greedy(sentence):
    """Translate one sentence with the library's greedy decoder."""
    ids = src_vocab.encode(sentence, add_special_tokens=True)
    tokens = library_greedy_decode(model, torch.tensor([ids]).to(device), max_len=20)[0]
    return tgt_vocab.decode(tokens, skip_special_tokens=True)


def library_beam(sentence, beam_size=3):
    """Translate one sentence with the library's beam search."""
    ids = src_vocab.encode(sentence, add_special_tokens=True)
    tokens = library_beam_search(
        model, torch.tensor([ids]).to(device), beam_size=beam_size, max_len=20
    )
    return tgt_vocab.decode(tokens, skip_special_tokens=True)


comparisons = 0
for sentence in test_sentences:
    ours = greedy_decode(model, sentence, src_vocab, tgt_vocab, device)
    theirs = library_greedy(sentence)
    assert ours == theirs, f"greedy disagrees on {sentence!r}: {ours!r} vs {theirs!r}"
    comparisons += 1

    for beam_size in (1, 3, 5):
        ours = beam_search_decode(
            model, sentence, src_vocab, tgt_vocab, device, beam_size=beam_size
        )
        theirs = library_beam(sentence, beam_size)
        assert ours == theirs, (
            f"beam-{beam_size} disagrees on {sentence!r}: {ours!r} vs {theirs!r}"
        )
        comparisons += 1

print(f"{comparisons} comparisons, notebook and library agree on every one.")

20 comparisons, notebook and library agree on every one.


Note what that does and does not show. The two implementations return the same
translations for *these* sentences and *this* model. They are not the same algorithm, as
the table above says, so use the library's version when the answer has to be right.

## Part 5: Scoring the two decoders

A decoding setting is chosen by scoring what it produces. What the scores measure, and how
to read one, is [Tutorial 2](02-evaluating-translations.ipynb)'s subject; here we only use
them.

Score with `torchlingo.evaluation`, and keep each score's **signature** next to it: the
signature records how the number was produced, so two scores can be compared only when
their signatures agree, just as two models can be compared only when they were decoded
the same way. These are three- and four-word phrases, so the score is chrF: BLEU's 4-grams
barely exist at this length.

In [14]:
from torchlingo.evaluation import compute_chrf

sources = ["where is the station", "one ticket please", "turn left here", "we need a taxi"]
references = ["dónde está la estación", "un billete por favor", "gira a la izquierda aquí",
              "necesitamos un taxi"]

greedy_translations = [greedy_decode(model, s, src_vocab, tgt_vocab, device) for s in sources]
beam_translations = [beam_search_decode(model, s, src_vocab, tgt_vocab, device) for s in sources]

greedy_chrf = compute_chrf(greedy_translations, references)
beam_chrf = compute_chrf(beam_translations, references)

print(f"chrF, greedy: {greedy_chrf.score:.1f}")
print(f"chrF, beam 3: {beam_chrf.score:.1f}")
print(f"signature:    {beam_chrf.signature}")

chrF, greedy: 100.0
chrF, beam 3: 100.0
signature:    nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.6.0


In [15]:
# Detailed comparison
print(f"{'Source':<22} {'Reference':<26} {'Greedy':<26} {'Beam':<26}")
print("-" * 100)

for src, ref, greedy, beam in zip(sources, references, greedy_translations, beam_translations):
    print(f"{src:<22} {ref:<26} {greedy:<26} {beam:<26}")

Source                 Reference                  Greedy                     Beam                      
----------------------------------------------------------------------------------------------------
where is the station   dónde está la estación     dónde está la estación     dónde está la estación    
one ticket please      un billete por favor       un billete por favor       un billete por favor      
turn left here         gira a la izquierda aquí   gira a la izquierda aquí   gira a la izquierda aquí  
we need a taxi         necesitamos un taxi        necesitamos un taxi        necesitamos un taxi       


**Don't read anything into those scores.** The four test phrases are four of the ten
the model trained on, so it has memorized them and the scores come out at 100. That shows
the scoring runs, not that the model translates: it has never seen a sentence it wasn't
trained on. A score means something only on a held-out test set, like the one A9 scores.

## Part 6: Drawing instead of choosing

Greedy and beam search both look for the most likely translation. Sometimes that is not
what you want. In **back-translation** (Lecture 13, A13) a reverse model translates text
that exists only in the target language, and the output becomes extra *source* sentences
for training. There, the most likely translation is too regular to teach the forward model
much.

Two strategies make deliberately varied output instead:

- **`"sample"`**: at each step, draw the next word at random, in proportion to its
  probability. A word the model gives 30% is drawn about 30% of the time. `temperature`
  divides the scores first: below 1 sharpens toward greedy, above 1 flattens toward
  uniform.
- **`"beam+noise"`**: beam search, then corrupt each output: delete each word with
  probability 0.1, replace each with `<blank>` with probability 0.1, and shuffle the
  words, moving none more than three places.

Both take a `seed`, so running the cell again gives the same draws. Six draws of one
sentence, at two temperatures, and four noised beams:

In [16]:
from torchlingo.inference import translate_batch

sentence = "the train is late"
for temperature in (1.0, 2.0):
    draws = translate_batch(model, [sentence] * 6, src_vocab, tgt_vocab,
                            decode_strategy="sample", temperature=temperature, seed=0,
                            max_len=20, device=device, config=config)
    print(f"sampled, temperature {temperature}:")
    for draw in draws:
        print(f"    {draw}")

noisy = translate_batch(model, [sentence] * 4, src_vocab, tgt_vocab,
                        decode_strategy="beam+noise", beam_size=3, seed=0,
                        max_len=20, device=device, config=config)
print("beam 3, then noise:")
for line in noisy:
    print(f"    {line}")

sampled, temperature 1.0:
    el tren llega tarde
    el tren llega tarde
    el tren llega tarde
    favor
    el tren llega tarde
    el tren está tren
sampled, temperature 2.0:
    el llega por tren billete
    <unk>
    el tren llega pesa estación mi llega tarde
    pesa
    el tren llega tarde
    el hotel


beam 3, then noise:
    el llega
    el tarde tren llega
    el tren tarde llega
    tren llega el tarde


At temperature 1 the model mostly writes the phrase it memorized, and now and then a draw
goes astray: a word it gave a few percent was drawn, and every later step had to continue
from it. At temperature 2 the strays outnumber the phrase. Noised beam starts from the
right answer every time and then breaks it, differently in each row.

As translations these are worse, which is why you never score a system on them. As
training sources they are what back-translation wants. Edunov et al. (2018) found that
synthetic sources made either way train a better forward model than beam or greedy ones,
in all but resource-poor settings, where the reverse model is too weak for its samples to
be worth much. A13 lets you test which side of that line your language is on.

## What to take away

You have written greedy decoding and beam search out in full, watched beam search prune,
checked both against `torchlingo.inference`, scored their output, and drawn translations
at random instead of choosing them.

**Decoding is part of the system you are measuring.** Change it and the score moves, so a
comparison between two models holds it fixed. A9 compares your A8 and A9 models with both
decoded greedily for exactly that reason; a BLEU difference between a greedy run and a
beam run says nothing about the models.

On a real model, which this tiny one is not, the settings behave like this (measured on
the pretrained model from [Tutorial 7](07-real-translations.ipynb); numbers and error bars
in [Decoding](../concepts/decoding.md)):

- **Beam size: most of the gain is the first step past greedy, and past the peak it
  reverses.** Greedy to any beam width is worth roughly 1.2 to 1.6 BLEU; beam 2 to 3 gains
  a little, 3 to 5 is flat, and 5 to 10 measurably *loses*, at about 7x the cost of beam 2.
- **Why widening hurts.** Beam search maximizes total log-probability, and every token
  adds a negative term, so a wider search finds shorter output: mean length falls from
  12.3 tokens at greedy to 9.7 at beam 10, against references averaging 11.6.
- **The length penalty, alpha,** is meant to correct exactly that, and on that model it
  does not: alpha from 0.0 to 1.0 is indistinguishable. Sweep it rather than trusting it.
- **Maximum length** cuts a translation off without saying so. Set it from your longest
  target sentence.
- **A flat sweep is a statement about the experiment** before it is one about the
  setting: first ask whether the experiment could have detected an effect.
- **The best translation is not always what you want.** For back-translation, sampling or
  noised beam output can train a better model than the best translations do; for scoring,
  never sample.
- **Use `torchlingo.inference` in real work.** The functions here are for reading. The
  library's break ties by a fixed rule, so output does not depend on the device; they
  normalize length while pruning; and `torchlingo.inference_fast` batches them, with
  identical output, much faster.